In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ===== 配置 =====
FILES = [Path("btc_2015.xlsx"), Path("btc_2021.xlsx"), Path("btc_2025.xlsx")]
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

# AHR999 参数（与原 Pine Script 一致）
LEN_GMA = 200                 # 200日几何均值
K, B = 5.84, -17.01           # 估值中枢：10^(K * log10(age) + B)
BITCOIN_BIRTH = pd.Timestamp("2009-01-03")  # 比特币生日

# ===== 工具函数 =====
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    """读取两列（时间、价格），支持‘横着两行’；返回按时间升序去重后的 DataFrame。"""
    assert fp.exists(), f"未找到文件：{fp}"
    df = pd.read_excel(fp, engine="openpyxl", header=None)
    if df.shape[0] <= 3 and df.shape[1] > df.shape[0]:
        df = df.T  # 横排两行 → 竖列
    df = df.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return df.dropna().drop_duplicates("ts").sort_values("ts")

def to_daily_close(df: pd.DataFrame) -> pd.Series:
    """转为日频‘收盘’（last），并前向填充补全日期。"""
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "date"
    return s

# ===== 1) 读取三个文件 → 合并成一条日线价格 =====
daily_series = []
for f in FILES:
    daily_series.append(to_daily_close(read_two_col_excel(f)))

# 合并（重叠日期取首个非空；理论上三表在重叠处价格相同）
df_all = pd.concat(daily_series, axis=1)
price = df_all.bfill(axis=1).iloc[:, 0].rename("price")  # 取每行第一个非空
price = price.sort_index()
assert not price.empty, "价格序列为空，请检查输入文件"

# ===== 2) 计算 AHR999 各组成项 =====
out = pd.DataFrame(index=price.index)
out["price"] = price.astype(float)

# 200日‘几何均值’（避免乘法溢出，用 log 的滚动平均）
logp = np.log(out["price"])
gma200 = np.exp(logp.rolling(LEN_GMA, min_periods=LEN_GMA).mean())
out["gma200"] = gma200

# 平均成本因子：avg_index = P / GMA200
out["avg_index"] = out["price"] / out["gma200"]



# 年龄估值中枢：estimate_price = 10^(K*log10(age)+B)
# 用“自 2009-01-03 起的天数”作为 age（浮点），并把非正数设为 NaN，避免 log10 出错
age_days = pd.Series(
    ((out.index - BITCOIN_BIRTH) / pd.Timedelta(days=1)).astype(float),
    index=out.index,
    dtype=float
)
age_days = age_days.where(age_days > 0, np.nan)

with np.errstate(divide="ignore", invalid="ignore"):
    estimate_price = np.power(10.0, K * np.log10(age_days) + B)

out["estimate_price"] = estimate_price
out["estimate_index"] = out["price"] / out["estimate_price"]
out["ahr999"] = out["avg_index"] * out["estimate_index"]




# ===== 3) 导出 =====
xlsx_path = OUTDIR / "ahr999_daily.xlsx"
csv_path  = OUTDIR / "ahr999_daily.csv"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
    out.to_excel(w, sheet_name="AHR999")

out.to_csv(csv_path, index_label="date", encoding="utf-8-sig")

print("已保存：", xlsx_path.resolve())
print("也导出：", csv_path.resolve())
print(out.tail(5))


C:\Users\nickc\AppData\Local\Temp\ipykernel_28732\1650054850.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_28732\1650054850.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


已保存： C:\Users\nickc\yanda\btc\btc_fit_outputs09142025\ahr999_daily.xlsx
也导出： C:\Users\nickc\yanda\btc\btc_fit_outputs09142025\ahr999_daily.csv
                price         gma200  avg_index  estimate_price  \
date                                                              
2025-09-10  111531.25  101091.078087   1.103275   124116.724933   
2025-09-11  113961.43  101174.771219   1.126382   124235.715674   
2025-09-12  115507.79  101266.932064   1.140627   124354.800943   
2025-09-13  116093.56  101387.889059   1.145044   124473.980798   
2025-09-14  115951.91  101524.146924   1.142112   124593.255300   

            estimate_index    ahr999  
date                                  
2025-09-10        0.898600  0.991402  
2025-09-11        0.917300  1.033230  
2025-09-12        0.928857  1.059479  
2025-09-13        0.932673  1.067952  
2025-09-14        0.930644  1.062899  


: 